In [2]:
import xarray as xr
import pandas as pd
import dask
import glob
import os
import math

In [3]:
import numpy as np
import re

In [4]:
from dask.distributed import Client

from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=32, threads_per_worker=2)
client = Client(cluster)
client

/glade/work/qingyuany/conda-envs/ml_env/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42137 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/qingyuany/Work/proxy/42137/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/qingyuany/Work/proxy/42137/status,Workers: 32
Total threads: 64,Total memory: 64.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41269,Workers: 32
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/qingyuany/Work/proxy/42137/status,Total threads: 64
Started: Just now,Total memory: 64.00 GiB
Comm: tcp://127.0.0.1:41681,Total threads: 2
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/qingyuany/Work/proxy/32957/status,Memory: 2.00 GiB
Nanny: tcp://127.0.0.1:39583,


In [5]:
v4_simu1 = xr.open_dataset("/glade/work/qingyuany/simulations/iterations/spatialtuning_v4/v4_clim_vars_1_20.nc")
v4_simu2 = xr.open_dataset("/glade/work/qingyuany/simulations/iterations/spatialtuning_v4/v4_clim_vars_21_40.nc")
v4_simu3 = xr.open_dataset("/glade/work/qingyuany/simulations/iterations/spatialtuning_v4/v4_clim_vars_41_60.nc")
v4_simu_yang = xr.open_dataset("/glade/work/qingyuany/simulations/iterations/spatialtuning_v4/v4_yang_clim_vars_1_30.nc")

sim_v4 = xr.concat([v4_simu1, v4_simu2, v4_simu3], dim = "ppe_ind")
sim_v4_yang = xr.concat([v4_simu_yang], dim = "ppe_ind")

In [6]:
sim_v4_yang["ppe_ind"] = sim_v4_yang["ppe_ind"] + 60
sim_v4_yang["ppe_ind"]

<xarray.DataArray 'ppe_ind' (ppe_ind: 30)> Size: 240B
array([61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78,
       79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90])
Coordinates:
  * ppe_ind  (ppe_ind) int64 240B 61 62 63 64 65 66 67 ... 84 85 86 87 88 89 90

In [7]:
sim_v4 = xr.concat([sim_v4, sim_v4_yang], dim = "ppe_ind")


In [8]:
sim_v4.ppe_ind

<xarray.DataArray 'ppe_ind' (ppe_ind: 87)> Size: 696B
array([ 1,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20,
       21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38,
       40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57,
       58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75,
       76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90])
Coordinates:
  * ppe_ind  (ppe_ind) int64 696B 1 3 4 5 6 7 8 9 10 ... 83 84 85 86 87 88 89 90

In [9]:
sim_v4["PRECT"] = sim_v4["PRECC"] + sim_v4["PRECL"]


In [10]:
obs_ds = xr.open_dataset("~/satellite_obs/obs_interp_cam6.nc")

%run ./common_vars.py

In [11]:
parav4 = pd.read_csv("~/cam_ml/paras/v4/v4_para_scale20_zonal_original_scale.csv", index_col = False)
parav4 = parav4.iloc[:60,]
parav4.index = parav4.index + 1

parav4  = parav4[parav4.index.isin(sim_v4.ppe_ind.values)]
parav4.index

Index([ 1,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20,
       21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38,
       40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57,
       58, 59, 60],
      dtype='int64')

In [12]:
parav4_yang = pd.read_csv("~/cam_ml/paras/v4_yang/v4_yang_scale20_zonal_original_scale.csv", index_col = False)
parav4_yang = parav4_yang.iloc[:30,]
parav4_yang.index = parav4_yang.index + 61
parav4_yang = parav4[parav4.index.isin(sim_v4.ppe_ind.values)]


In [13]:

lat_bins = np.arange(-85, 86, 5)  # -90 to 90 every 10 degrees
lat_labels = ((lat_bins[:-1] + lat_bins[1:]) / 2).astype(int).astype(str)  # midpoint labels
lat_labels
lat_labels = np.char.add("zonal_lat_",lat_labels)
lat_labels

array(['zonal_lat_-82', 'zonal_lat_-77', 'zonal_lat_-72', 'zonal_lat_-67',
       'zonal_lat_-62', 'zonal_lat_-57', 'zonal_lat_-52', 'zonal_lat_-47',
       'zonal_lat_-42', 'zonal_lat_-37', 'zonal_lat_-32', 'zonal_lat_-27',
       'zonal_lat_-22', 'zonal_lat_-17', 'zonal_lat_-12', 'zonal_lat_-7',
       'zonal_lat_-2', 'zonal_lat_2', 'zonal_lat_7', 'zonal_lat_12',
       'zonal_lat_17', 'zonal_lat_22', 'zonal_lat_27', 'zonal_lat_32',
       'zonal_lat_37', 'zonal_lat_42', 'zonal_lat_47', 'zonal_lat_52',
       'zonal_lat_57', 'zonal_lat_62', 'zonal_lat_67', 'zonal_lat_72',
       'zonal_lat_77', 'zonal_lat_82'], dtype='<U31')

In [14]:
len(lat_labels)

34

In [15]:
ppe_zonal_list = []
obs_zonal_list = []

for cam_name, obs_name in obs_dict.items():

    ppes = sim_v4[cam_name]

    filter_tf = obs_ds[obs_name].notnull()
    ppes_filtered = ppes.where(filter_tf)

    
    zonal_ppe_temp = ppes_filtered.mean(dim  = "lon", skipna = True).groupby_bins("lat",lat_bins, labels = lat_labels).mean(dim = "lat", skipna = True).to_dataframe().unstack(level = 1)
    zonal_ppe_temp.columns.name = None
    zonal_ppe_temp.columns = ["_".join(col) for col in list(zonal_ppe_temp.columns)]

    
    zonal_obs_temp = obs_ds[obs_name].mean(dim = "lon", skipna = True).groupby_bins("lat",lat_bins, labels = lat_labels).mean(dim = "lat", skipna = True).to_series()
    zonal_obs_temp.index = zonal_ppe_temp.columns

    
    ppe_zonal_list.append(zonal_ppe_temp)
    obs_zonal_list.append(zonal_obs_temp)
    

In [16]:
ppe_zonal_pd = pd.concat(ppe_zonal_list, axis = 1)
obs_zonal_pd = pd.concat(obs_zonal_list)


In [17]:
### Added locations for SWCF, LWCF, and PRECT
man_sel_locations1 = pd.Series({"cli": "SWCF", "lat_min": -6,"lat_max": -4, "lon_min":  141, "lon_max": 144})
man_sel_locations2 = pd.Series({"cli": "SWCF", "lat_min": 7,"lat_max": 9, "lon_min":  235, "lon_max": 240})
man_sel_locations3 = pd.Series({"cli": "SWCF", "lat_min": -24,"lat_max": -22, "lon_min":  360-77, "lon_max": 360-72})

man_sel_locations4 = pd.Series({"cli": "LWCF", "lat_min": 6,"lat_max": 8, "lon_min":  275, "lon_max": 280})
man_sel_locations5 = pd.Series({"cli": "LWCF", "lat_min": 0,"lat_max": 5, "lon_min":  80, "lon_max": 90})
man_sel_locations6 = pd.Series({"cli": "LWCF", "lat_min": 5,"lat_max": 10, "lon_min":  200, "lon_max": 220})

man_sel_locations7= pd.Series({"cli": "TMQ", "lat_min": 5,"lat_max": 10, "lon_min":  200, "lon_max": 220})
man_sel_locations8 = pd.Series({"cli": "TMQ", "lat_min": 5,"lat_max": 8, "lon_min":  135, "lon_max": 145})
man_sel_locations9 = pd.Series({"cli": "TMQ", "lat_min": 10,"lat_max": 15, "lon_min":  90, "lon_max": 95})

man_sel_locations10 = pd.Series({"cli": "FLUT", "lat_min": 6,"lat_max": 8, "lon_min":  275, "lon_max": 280})
man_sel_locations11 = pd.Series({"cli": "FLUT", "lat_min": 0,"lat_max": 5, "lon_min":  85, "lon_max": 90})

man_sel_locations12 = pd.Series({"cli": "PRECT", "lat_min": 6,"lat_max": 8, "lon_min":  275, "lon_max": 280})
man_sel_locations13 = pd.Series({"cli": "PRECT", "lat_min": 6,"lat_max": 10, "lon_min":  220, "lon_max": 240})
man_sel_locations14 = pd.Series({"cli": "PRECT", "lat_min": 0,"lat_max": 5, "lon_min":  90, "lon_max": 95})
man_sel_locations15 = pd.Series({"cli": "PRECT", "lat_min": -1,"lat_max": 1, "lon_min":  113, "lon_max": 115})



manul_ppe_info = pd.concat([man_sel_locations1, man_sel_locations2, man_sel_locations3, man_sel_locations4, 
                            man_sel_locations5, man_sel_locations6, man_sel_locations7, man_sel_locations8,
                            man_sel_locations9, man_sel_locations10, man_sel_locations11, man_sel_locations12, 
                            man_sel_locations13, man_sel_locations14, man_sel_locations15], axis  = 1).transpose()

In [18]:
manul_ppe_info

,cli,lat_min,lat_max,lon_min,lon_max
0,SWCF,-6,-4,141,144
1,SWCF,7,9,235,240
2,SWCF,-24,-22,283,288
3,LWCF,6,8,275,280
4,LWCF,0,5,80,90
5,LWCF,5,10,200,220
6,TMQ,5,10,200,220
7,TMQ,5,8,135,145
8,TMQ,10,15,90,95
9,FLUT,6,8,275,280


In [19]:
ppe_manual_list = []
obs_manual_list = []
manual_name_list = []

for row_ind, row in manul_ppe_info.iterrows():
    
    temp_obs = obs_ds[obs_dict[row.cli]].sel(lat = slice(row.lat_min, row.lat_max), lon = slice(row.lon_min, row.lon_max)).mean(dim = ["lat", "lon"]).values
    if ~np.isnan(temp_obs):
        temp_ppe = sim_v4[row.cli].sel(lat = slice(row.lat_min, row.lat_max), lon = slice(row.lon_min, row.lon_max)).mean(dim = ["lat", "lon"]).to_dataframe()
        
        manual_name_list.append("_".join(row.astype(str)))
        ppe_manual_list.append(temp_ppe)
        obs_manual_list.append(temp_obs)
    else:
        print("??")


In [20]:
ppe_manual_list = pd.concat(ppe_manual_list,axis = 1)
obs_manual_list = pd.Series(obs_manual_list)

ppe_manual_list.columns = manual_name_list
obs_manual_list.index = manual_name_list

In [21]:
ppe_zonal_manulselect = pd.concat([ppe_zonal_pd, ppe_manual_list], axis = 1)
obs_zonal_manulselect = pd.concat([obs_zonal_pd, obs_manual_list])


In [22]:
ppe_zonal_manulselect

,SWCF_zonal_lat_-82,SWCF_zonal_lat_-77,SWCF_zonal_lat_-72,SWCF_zonal_lat_-67,SWCF_zonal_lat_-62,SWCF_zonal_lat_-57,SWCF_zonal_lat_-52,SWCF_zonal_lat_-47,SWCF_zonal_lat_-42,SWCF_zonal_lat_-37,...,LWCF_5_10_200_220,TMQ_5_10_200_220,TMQ_5_8_135_145,TMQ_10_15_90_95,FLUT_6_8_275_280,FLUT_0_5_85_90,PRECT_6_8_275_280,PRECT_6_10_220_240,PRECT_0_5_90_95,PRECT_-1_1_113_115
ppe_ind,,,,,,,,,,,,,,,,,,,,,
1,-1.775323,-3.677040,-8.188253,-26.007094,-55.478150,-73.107625,-77.024634,-73.212884,-64.607686,-54.185431,...,42.237118,50.926451,51.155686,48.030842,188.725400,235.855198,2.144443e-07,1.291109e-07,5.737806e-08,1.112224e-07
3,-1.618221,-3.374800,-8.302124,-26.239062,-55.215606,-72.102811,-75.852205,-72.355755,-63.536108,-53.471915,...,40.278538,49.468480,47.521801,42.907259,205.532444,248.364480,2.203473e-07,1.355944e-07,6.664543e-08,1.414240e-07
4,-1.765804,-3.844527,-8.594140,-27.004102,-56.849576,-74.685015,-78.965654,-77.357821,-74.099307,-67.848767,...,42.208071,52.102258,52.861437,47.787588,209.736810,241.644955,1.721039e-07,1.397144e-07,5.880796e-08,1.355674e-07
5,-2.353393,-4.225666,-8.897521,-27.351285,-56.519404,-73.319435,-76.810730,-74.351819,-69.418165,-63.010395,...,40.124951,51.354876,54.582372,49.958684,189.795158,225.372543,1.737110e-07,1.020140e-07,5.462205e-08,1.159605e-07
6,-1.377978,-3.060643,-7.356543,-24.552219,-52.189145,-68.319704,-71.086715,-68.272482,-62.101780,-52.859389,...,31.107965,51.758989,54.698607,49.581895,220.764464,241.434916,1.176212e-07,9.157055e-08,5.267397e-08,1.196123e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,-1.606898,-3.169294,-7.590625,-24.659385,-52.536696,-70.074936,-74.534792,-72.147591,-66.729659,-58.851437,...,46.998606,50.305236,51.149755,46.134638,192.510564,245.196242,2.171843e-07,1.319031e-07,3.920592e-08,1.056178e-07
87,-1.762400,-3.385148,-8.145162,-25.911908,-53.455347,-69.577070,-72.606119,-67.930465,-58.418597,-47.882339,...,41.495487,50.062595,52.688789,47.775832,204.089257,244.748199,1.519858e-07,1.051643e-07,3.953457e-08,1.130715e-07
88,-2.018235,-3.873849,-9.167514,-27.983393,-57.934004,-76.453763,-82.638755,-82.466202,-79.755906,-74.506072,...,45.953280,52.789861,54.100072,48.664743,204.380007,230.998497,1.722230e-07,1.310283e-07,6.062982e-08,1.251549e-07


In [23]:
ppe_zonal_manulselect.to_csv("/glade/work/qingyuany/camml_v6/zonal_manual_climatologies/ppe_zonal_manualselect.csv", index = True)
obs_zonal_manulselect.to_csv("/glade/work/qingyuany/camml_v6/zonal_manual_climatologies/obs_zonal_manualselect.csv", index = True)
